In [ ]:
print(spark.version)

## Global Settings

In [ ]:
import pyspark.sql.types as T
import pyspark.sql.functions as F
    
KM_TO_MILE = 0.6214

GREEN_TAXI_S3_PATH = "s3://robot-dreams-source-data/home-work-1-unified/nyc_taxi/green/"
YELLOW_TAXI_S3_PATH = "s3://robot-dreams-source-data/home-work-1-unified/nyc_taxi/yellow/"
TAXI_ZONE_LOOKUP_S3_PATH = "s3://robot-dreams-source-data/home-work-1-unified/nyc_taxi/taxi_zone_lookup.csv"

from datetime import date

def get_current_date_string():
    return date.today().strftime("%Y-%m-%d")

ZONE_STATISTICS_S3_PATH = f"s3://aoliinyk-hw/results/zonestatstic/{get_current_date_string()}/zone_statistic.parquet"
ZONE_DAYS_STATISTICS_S3_PATH = f"s3://aoliinyk-hw/results/zone_days_statstic/{get_current_date_string()}/zone_days_statstic.parquet"

## Завдання 2: Імпорт, уніфікація та об’єднання

### 2.1. Імпорт даних та уніфікація схеми для green taxi

In [ ]:
raw_green_schema = T.StructType([
    T.StructField('VendorID', T.LongType(), True),
    T.StructField('passenger_count', T.LongType(), True),
    T.StructField('trip_distance', T.DoubleType(), True),
    T.StructField('RatecodeID', T.LongType(), True),
    T.StructField('store_and_fwd_flag', T.StringType(), True),
    T.StructField('PULocationID', T.LongType(), True),
    T.StructField('DOLocationID', T.LongType(), True),
    T.StructField('payment_type', T.LongType(), True),
    T.StructField('fare_amount', T.DoubleType(), True),
    T.StructField('extra', T.DoubleType(), True),
    T.StructField('mta_tax', T.DoubleType(), True),
    T.StructField('tip_amount', T.DoubleType(), True),
    T.StructField('tolls_amount', T.DoubleType(), True),
    T.StructField('improvement_surcharge', T.DoubleType(), True),
    T.StructField('total_amount', T.DoubleType(), True),
    T.StructField('congestion_surcharge', T.DoubleType(), True),
    T.StructField('tpep_pickup_datetime', T.TimestampNTZType(), True),
    T.StructField('tpep_dropoff_datetime', T.TimestampNTZType(), True),
    T.StructField('airport_fee', T.DoubleType(), True)
])

raw_green_df = (
    spark.read
        .option("recursiveFileLookup", "true")
        .schema(raw_green_schema)
        .parquet(GREEN_TAXI_S3_PATH)
)

unified_green_df = raw_green_df.withColumn("taxi_type", F.lit("green").cast(T.StringType()))

### 2.2. Імпорт даних та уніфікація схеми для yellow taxi

In [2]:
raw_yellow_schema = T.StructType([
    T.StructField('VendorID', T.LongType(), True),
    T.StructField('passenger_count', T.LongType(), True),
    T.StructField('trip_distance', T.DoubleType(), True),
    T.StructField('RatecodeID', T.LongType(), True),
    T.StructField('store_and_fwd_flag', T.StringType(), True),
    T.StructField('PULocationID', T.LongType(), True),
    T.StructField('DOLocationID', T.LongType(), True),
    T.StructField('payment_type', T.LongType(), True),
    T.StructField('fare_amount', T.DoubleType(), True),
    T.StructField('extra', T.DoubleType(), True),
    T.StructField('mta_tax', T.DoubleType(), True),
    T.StructField('tip_amount', T.DoubleType(), True),
    T.StructField('tolls_amount', T.DoubleType(), True),
    T.StructField('improvement_surcharge', T.DoubleType(), True),
    T.StructField('total_amount', T.DoubleType(), True),
    T.StructField('congestion_surcharge', T.DoubleType(), True),
    T.StructField('tpep_pickup_datetime', T.TimestampNTZType(), True),
    T.StructField('tpep_dropoff_datetime', T.TimestampNTZType(), True),
    T.StructField('airport_fee', T.DoubleType(), True)
])

raw_yellow_df = (
    spark.read
        .option("recursiveFileLookup", "true")
        .schema(raw_yellow_schema)
        .parquet(YELLOW_TAXI_S3_PATH)
)

unified_yellow_df = raw_yellow_df.withColumn("taxi_type", F.lit("yellow").cast(T.StringType()))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
name 'T' is not defined
Traceback (most recent call last):
NameError: name 'T' is not defined



### 2.3. Об’єднанання даних green and yellow taxis

##### 2.3.1. Об’єднанання даних green and yellow taxis.

In [ ]:
joined_trips_df = unified_green_df.unionByName(unified_yellow_df)

##### 2.3.2 Перейменування та додавання колонок

In [ ]:
schema_ready_trips_df = (
    joined_trips_df
        .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
        .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("pickup_hour", F.hour("pickup_datetime"))
        .withColumn("pickup_day_of_week", F.date_format(F.col("pickup_datetime"), "EEEE"))
        .withColumn("duration_min", F.expr("DATEDIFF(MINUTE, pickup_datetime, dropoff_datetime)"))
)

##### 2.3.3. Фільтрація аномальних записів та створення фінального trips_df

In [ ]:
trips_df = (
    schema_ready_trips_df
        .where(F.col("trip_distance") >= 0.1 * KM_TO_MILE)
        .where(F.col("fare_amount") > 2.0)
        .where(F.col("duration_min") >= 1)
)

### 2.6. JOIN між raw_trips_df і taxi_zone_lookup.csv з додаванням колонок pickup_zone, dropoff_zone

##### 2.6.1 Загрузити taxi_zone_lookup_df та додати колонки PULocationID, DOLocationID, pickup_zone, dropoff_zone

In [ ]:
taxi_zone_lookup_schema = T.StructType([
    T.StructField('LocationID', T.LongType(), True),
    T.StructField('Borough', T.StringType(), True),
    T.StructField('Zone', T.StringType(), True),
    T.StructField('service_zone', T.StringType(), True),
])

raw_taxi_zone_lookup_df = (
    spark.read
        .option("header", "true")
        .schema(taxi_zone_lookup_schema)
        .csv(TAXI_ZONE_LOOKUP_S3_PATH)
)

taxi_zone_lookup_df = (
    raw_taxi_zone_lookup_df
        .withColumn("PULocationID", F.col("LocationID"))
        .withColumn("DOLocationID", F.col("LocationID"))
        .withColumn("pickup_zone", F.col("Zone"))
        .withColumn("dropoff_zone", F.col("Zone"))
)

##### 2.6.2 Здійснити JOIN між trips_df та taxi_zone_lookup_df

In [ ]:
joined_trips_df = (
    trips_df
        .join(F.broadcast(taxi_zone_lookup_df), on="PULocationID", how="left")
        .join(F.broadcast(taxi_zone_lookup_df), on="DOLocationID", how="left")
        .cache()
)

## Завдання 3: Створення zone_summary

In [ ]:
zone_summary_df = (
    joined_trips_df
        .groupBy("pickup_zone")
        .agg(
            F.count("*").alias("total_trips"),
            (F.sum(F.when(F.col("taxi_type") == "green", 1).otherwise(0)) / F.count("*")).alias("green_share"),
            (F.sum(F.when(F.col("taxi_type") == "yellow", 1).otherwise(0)) / F.count("*")).alias("yellow_share"),
            F.avg("trip_distance").alias("avg_trip_distance"),
            F.max("trip_distance").alias("max_trip_distance"),
            F.avg("total_amount").alias("avg_total_amount"),
            F.avg("tip_amount").alias("avg_tip_amount"),
            F.min("tip_amount").alias("min_tip_amount")
        )
)

(
    zone_summary_df.write
        .mode("overwrite")
        .parquet(ZONE_STATISTICS_S3_PATH)
)

## Завдання 4: Створення zone_days_statstic

In [ ]:
zone_days_statstic_df = (
    joined_trips_df
        .groupBy("pickup_zone", "pickup_day_of_week")
        .agg(
            F.count("*").alias("total_trips"),
            # F.sum(F.when(F.col("fare_amount") > 30.0, 1).otherwise(0)) / F.count("*")).alias("high_fare_share"),
            F.expr("COUNT(CASE WHEN fare_amount > 30.0 THEN 1 END) / COUNT(*)").alias("high_fare_share")
        )
)

(
    zone_days_statstic_df.write
        .mode("overwrite")
        .parquet(ZONE_DAYS_STATISTICS_S3_PATH)
)